<a href="https://colab.research.google.com/github/danifer47551989/infovis/blob/main/Transformaciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import os
import re
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd


# =========================================================
# 1) CONFIGURACIÓN
# =========================================================
# En Colab, si tus JSON están en la misma carpeta visible del panel Files,
# podés dejar BASE_PATH = "."
BASE_PATH = "."

OUTPUT_CSV = "conversations_aggregated_for_viz.csv"
OUTPUT_XLSX = "conversations_aggregated_for_viz.xlsx"
OUTPUT_JSON = "conversations_aggregated_for_viz.json"


# =========================================================
# 2) REGLAS DE CATEGORIZACIÓN
# =========================================================
CATEGORY_MAP = {
    "AWS / Cloud": [
        "aws", "s3", "glue", "redshift", "emr", "athena", "lambda",
        "step functions", "kinesis", "dynamodb", "iam", "ec2",
        "cloudwatch", "boto3"
    ],
    "Data Engineering": [
        "airflow", "databricks", "delta lake", "lakehouse", "spark",
        "pyspark", "etl", "elt", "pipeline", "parquet", "orchestration",
        "dbt", "hudi", "iceberg"
    ],
    "SQL / BI": [
        "sql", "ssis", "ssrs", "power bi", "dax", "postgres",
        "postgresql", "snowflake", "tableau", "stored procedure",
        "cte", "join", "query", "semantic model"
    ],
    "Python / Programming": [
        "python", "pytest", "flask", "api", "oop", "docker",
        "kubernetes", "terraform", "scala", "java", "algorithm", "json"
    ],
    "Machine Learning / AI": [
        "machine learning", "xgboost", "lightgbm", "svm", "regression",
        "classification", "pytorch", "llm", "openai", "chatgpt",
        "feature engineering", "model"
    ],
    "Academic / Study": [
        "ejercicio", "tp", "trabajo practico", "parcial", "facultad",
        "clase", "universidad", "informe", "resumen", "teorico", "estadistica"
    ],
    "Personal / Advice": [
        "divorcio", "embarazada", "embarazo", "visa", "eb2", "eb-2",
        "plan de ahorro", "miami", "sueldo", "salario", "vecino",
        "parrillada", "outlook", "mail"
    ],
    "Translation / Writing": [
        "traduc", "translate", "redact", "email", "ingles", "english",
        "mensaje", "parrafo", "rewrite", "cover letter"
    ],
}

TYPE_MAP = {
    "Troubleshooting": [
        "error", "problema", "falla", "bug", "exception", "traceback",
        "issue", "fix", "debug", "warning", "failed", "not working",
        "no me", "no funciona"
    ],
    "How-to / Question": [
        "como ", "how ", "what ", "que ", "cual ", "can i", "puedo",
        "sirve", "difference", "diferencia", "what is", "que es",
        "me explicas"
    ],
    "Build / Design": [
        "armar", "build", "design", "arquitectura", "crear", "implement",
        "pagina", "dashboard", "modelo", "dataset", "estructura", "pipeline"
    ],
    "Analysis / Research": [
        "analiza", "analyze", "compare", "pros", "cons", "conviene",
        "beneficio", "impacto", "timeline", "investigar", "busca", "look up"
    ],
    "Writing / Translation": [
        "traduc", "translate", "redact", "rewrite", "email", "mensaje",
        "parrafo", "ingles", "english", "write"
    ],
    "Interview Prep / Learning": [
        "entrevista", "interview", "preguntas", "questions", "estudiar",
        "study", "practice", "explicame", "resumen", "learn"
    ],
}

TOOL_MAP = {
    "AWS Glue": ["glue"],
    "Apache Spark": ["spark", "pyspark"],
    "Apache Airflow": ["airflow"],
    "Amazon S3": ["s3"],
    "Amazon Redshift": ["redshift"],
    "Databricks": ["databricks", "delta lake", "unity catalog"],
    "Power BI": ["power bi", "dax"],
    "SQL Server / SSIS / SSRS": ["ssis", "ssrs", "sql server"],
    "dbt": ["dbt"],
    "Docker": ["docker", "dev container", "devcontainer"],
    "Terraform": ["terraform"],
    "Python": ["python", "pytest", "flask"],
    "R": [" r ", "rstudio", "r markdown", "ggplot"],
    "OpenAI / ChatGPT": ["openai", "chatgpt", "gpt-4", "gpt-5"],
    "Git / GitHub": ["github", "git "],
}


# =========================================================
# 3) FUNCIONES AUXILIARES
# =========================================================
def ts_to_dt(ts: Any) -> Optional[datetime]:
    if ts in (None, ""):
        return None
    try:
        return datetime.fromtimestamp(float(ts))
    except Exception:
        return None


def extract_text_from_content(content: Dict[str, Any]) -> str:
    """
    Extrae texto del campo content del mensaje.
    Soporta varios content_type del export de ChatGPT.
    """
    if not isinstance(content, dict):
        return ""

    content_type = content.get("content_type")

    if content_type in (
        "text",
        "code",
        "tether_quote",
        "reasoning_recap",
        "system_error",
        "execution_output"
    ):
        parts = content.get("parts", [])
        extracted = []

        for part in parts:
            if isinstance(part, str):
                extracted.append(part)
            elif isinstance(part, dict):
                for key in ("text", "content", "title", "result", "query"):
                    if key in part and isinstance(part[key], str):
                        extracted.append(part[key])

        if content_type == "reasoning_recap" and isinstance(content.get("content"), str):
            extracted.append(content["content"])

        return "\n".join(x for x in extracted if x)

    if content_type == "multimodal_text":
        parts = content.get("parts", [])
        extracted = []

        for part in parts:
            if isinstance(part, str):
                extracted.append(part)
            elif isinstance(part, dict):
                if isinstance(part.get("text"), str):
                    extracted.append(part["text"])

        return "\n".join(x for x in extracted if x)

    return ""


def classify_by_keywords(
    text: str,
    mapping: Dict[str, List[str]],
    default: str
) -> Tuple[str, Dict[str, int]]:
    """
    Clasifica un texto según coincidencias por palabras clave.
    Devuelve:
    - etiqueta ganadora
    - detalle de scores por etiqueta
    """
    clean_text = f" {text.lower()} "
    scores: Dict[str, int] = {}

    for label, keywords in mapping.items():
        score = sum(1 for kw in keywords if kw in clean_text)
        if score > 0:
            scores[label] = score

    if not scores:
        return default, {}

    best_label = sorted(scores.items(), key=lambda x: (-x[1], x[0]))[0][0]
    return best_label, scores


def top_tools(text: str) -> List[str]:
    """
    Detecta hasta 3 tools principales.
    """
    clean_text = f" {text.lower()} "
    found: List[Tuple[str, int]] = []

    for label, keywords in TOOL_MAP.items():
        score = sum(1 for kw in keywords if kw in clean_text)
        if score > 0:
            found.append((label, score))

    found = sorted(found, key=lambda x: (-x[1], x[0]))
    return [label for label, _ in found[:3]]


def fallback_tool_from_category(category: str) -> str:
    """
    Si no se detecta una tool concreta, se asigna una etiqueta
    más explicativa a partir de la categoría.
    """
    fallback = {
        "AWS / Cloud": "Cloud / AWS",
        "Data Engineering": "Data Pipeline / Engineering",
        "SQL / BI": "SQL / BI Tools",
        "Python / Programming": "Python / Programming",
        "Machine Learning / AI": "AI / ML",
        "Academic / Study": "Study / Coursework",
        "Personal / Advice": "Personal / Daily Life",
        "Translation / Writing": "Writing / Translation",
        "General / Other": "General Purpose"
    }
    return fallback.get(category, "General Purpose")


def list_conversation_files(base_path: str) -> List[str]:
    """
    Lista todos los conversations-xxx.json de la carpeta.
    """
    files = [
        f for f in os.listdir(base_path)
        if re.match(r"conversations-\d+\.json$", f)
    ]
    return sorted(files)


# =========================================================
# 4) CARGA DE LOS JSON
# =========================================================
conversation_files = list_conversation_files(BASE_PATH)

if not conversation_files:
    raise FileNotFoundError(
        "No se encontraron archivos conversations-xxx.json en la carpeta actual."
    )

print("Archivos detectados:")
for f in conversation_files:
    print(" -", f)

all_conversations: List[Dict[str, Any]] = []

for filename in conversation_files:
    full_path = os.path.join(BASE_PATH, filename)
    with open(full_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        all_conversations.extend(data)

print(f"\nTotal de conversaciones cargadas: {len(all_conversations)}")


# =========================================================
# 5) TRANSFORMACIÓN A DATASET FINAL
# =========================================================
rows: List[Dict[str, Any]] = []

for conv in all_conversations:
    conversation_id = conv.get("conversation_id") or conv.get("id")
    title = conv.get("title") or ""
    model_slug = conv.get("default_model_slug") or ""

    messages: List[Dict[str, Any]] = []

    mapping = conv.get("mapping", {})
    for node in mapping.values():
        msg = node.get("message")
        if not msg:
            continue

        role = msg.get("author", {}).get("role")

        # Solo user/assistant
        if role not in ("user", "assistant"):
            continue

        text = extract_text_from_content(msg.get("content", {}))
        dt = ts_to_dt(msg.get("create_time"))

        messages.append({
            "role": role,
            "datetime": dt,
            "text": text,
            "text_len": len(text or "")
        })

    # Ordenar cronológicamente
    messages = sorted(
        messages,
        key=lambda x: (x["datetime"] is None, x["datetime"] or datetime.max)
    )

    user_messages = [
        m for m in messages
        if m["role"] == "user" and (m["text"] or "").strip()
    ]
    assistant_messages = [
        m for m in messages
        if m["role"] == "assistant" and (m["text"] or "").strip()
    ]

    from_dt = min(
        (m["datetime"] for m in messages if m["datetime"] is not None),
        default=None
    )
    to_dt = max(
        (m["datetime"] for m in messages if m["datetime"] is not None),
        default=None
    )

    # Texto combinado para clasificar:
    # título + primeros mensajes del usuario
    combined_text = " \n ".join(
        [title] + [m["text"] for m in user_messages[:20]]
    )[:60000]

    primary_category, category_scores = classify_by_keywords(
        combined_text,
        CATEGORY_MAP,
        "General / Other"
    )

    primary_type, type_scores = classify_by_keywords(
        combined_text,
        TYPE_MAP,
        "General Inquiry"
    )

    tools = top_tools(combined_text)
    primary_tool = tools[0] if tools else fallback_tool_from_category(primary_category)
    secondary_tool = tools[1] if len(tools) > 1 else ""
    tertiary_tool = tools[2] if len(tools) > 2 else ""

    total_user_chars = sum(m["text_len"] for m in user_messages)
    user_count = len(user_messages)

    # Complejidad heurística
    if total_user_chars < 250 and user_count <= 2:
        complexity_bucket = "Low"
    elif total_user_chars < 1500 and user_count <= 6:
        complexity_bucket = "Medium"
    else:
        complexity_bucket = "High"

    # Was solved heurístico:
    # si después del último mensaje del usuario hay al menos una respuesta del assistant
    if user_messages and assistant_messages:
        last_user_dt = max(
            (m["datetime"] for m in user_messages if m["datetime"] is not None),
            default=None
        )
        was_solved = True if last_user_dt is None else any(
            (m["datetime"] is not None and m["datetime"] >= last_user_dt)
            for m in assistant_messages
        )
    else:
        was_solved = bool(assistant_messages)

    duration_minutes = round(
        (to_dt - from_dt).total_seconds() / 60, 2
    ) if from_dt and to_dt else 0

    rows.append({
        "conversation_id": conversation_id,
        "title": title,
        "date": from_dt.date().isoformat() if from_dt else "",
        "from": from_dt.isoformat(sep=" ") if from_dt else "",
        "to": to_dt.isoformat(sep=" ") if to_dt else "",
        "duration_minutes": duration_minutes,
        "message_count_total": len(messages),
        "user_message_count": len(user_messages),
        "assistant_message_count": len(assistant_messages),
        "primary_category": primary_category,
        "primary_type": primary_type,
        "primary_tool": primary_tool,
        "secondary_tool": secondary_tool,
        "tertiary_tool": tertiary_tool,
        "was_solved": was_solved,
        "complexity_bucket": complexity_bucket,
        "avg_user_message_length": round(total_user_chars / user_count, 1) if user_count else 0,
        "model_slug": model_slug
    })

df = pd.DataFrame(rows)
df = df.sort_values(["from", "conversation_id"], na_position="last").reset_index(drop=True)

print("\nDataset final generado.")
print("Filas:", len(df))
print("Columnas:", list(df.columns))


# =========================================================
# 6) EXPORTACIÓN
# =========================================================
df.to_csv(OUTPUT_CSV, index=False)
df.to_excel(OUTPUT_XLSX, index=False)

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(df.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

print("\nArchivos exportados:")
print(" -", OUTPUT_CSV)
print(" -", OUTPUT_XLSX)
print(" -", OUTPUT_JSON)


# =========================================================
# 7) PREVIEW
# =========================================================
display(df.head(10))
print("\nTop 10 primary_category:")
print(df["primary_category"].value_counts().head(10))

print("\nTop 10 primary_type:")
print(df["primary_type"].value_counts().head(10))

print("\nTop 10 primary_tool:")
print(df["primary_tool"].value_counts().head(10))

Archivos detectados:
 - conversations-000.json
 - conversations-001.json
 - conversations-002.json
 - conversations-003.json
 - conversations-004.json
 - conversations-005.json
 - conversations-006.json
 - conversations-007.json
 - conversations-008.json
 - conversations-009.json
 - conversations-010.json
 - conversations-011.json
 - conversations-012.json

Total de conversaciones cargadas: 1255

Dataset final generado.
Filas: 1255
Columnas: ['conversation_id', 'title', 'date', 'from', 'to', 'duration_minutes', 'message_count_total', 'user_message_count', 'assistant_message_count', 'primary_category', 'primary_type', 'primary_tool', 'secondary_tool', 'tertiary_tool', 'was_solved', 'complexity_bucket', 'avg_user_message_length', 'model_slug']

Archivos exportados:
 - conversations_aggregated_for_viz.csv
 - conversations_aggregated_for_viz.xlsx
 - conversations_aggregated_for_viz.json


,conversation_id,title,date,from,to,duration_minutes,message_count_total,user_message_count,assistant_message_count,primary_category,primary_type,primary_tool,secondary_tool,tertiary_tool,was_solved,complexity_bucket,avg_user_message_length,model_slug
0,781bddbb-de23-4da0-8901-ab101c46f877,SQL Query Optimization Help,2023-02-10,2023-02-10 19:52:13.645791,2023-03-15 14:57:45.986008,47225.54,12,6,6,SQL / BI,How-to / Question,R,SQL Server / SSIS / SSRS,,True,High,456.2,
1,c1556289-d972-444f-bf33-5d602af377f0,Slow Query Optimization Tips,2023-02-10,2023-02-10 19:53:15.903051,2023-02-10 20:18:07.230555,24.86,15,7,8,SQL / BI,Troubleshooting,SQL / BI Tools,,,True,High,1910.3,
2,db498881-72cb-4979-9e57-39642b404ed1,SQL Query Help Needed,2023-02-10,2023-02-10 19:54:52.780199,2023-02-10 19:55:09.611090,0.28,2,1,1,SQL / BI,General Inquiry,SQL / BI Tools,,,True,Low,33.0,
3,39d67f70-c0e3-4271-83b0-a6d78646dc74,Create Button In Python Web,2023-02-13,2023-02-13 11:21:07.274784,2023-02-13 11:22:07.501984,1.00,2,1,1,Python / Programming,Build / Design,Python,,,True,Low,43.0,
4,c580af00-4ace-43f1-8996-ac4e99f70102,Join varchar columns possible,2023-02-13,2023-02-13 14:26:53.694294,2023-02-13 14:27:15.307315,0.36,2,1,1,SQL / BI,General Inquiry,SQL / BI Tools,,,True,Low,29.0,
5,1eb95901-e612-4a88-94d5-57e54f33cc67,Removing Duplicates in Join,2023-02-13,2023-02-13 14:29:55.424565,2023-02-13 14:30:56.264282,1.01,2,1,1,SQL / BI,General Inquiry,SQL / BI Tools,,,True,Low,29.0,
6,d89c64de-e71b-4f2b-b341-b7244a74002a,Assist User Request,2023-02-14,2023-02-14 12:56:45.708909,2023-02-14 12:56:47.188582,0.02,2,1,1,SQL / BI,General Inquiry,SQL Server / SSIS / SSRS,,,True,Low,2.0,
7,fd9ee25c-4fcd-4290-97d0-9832a80c1183,New chat,2023-02-14,2023-02-14 12:57:38.560248,2023-02-14 16:46:41.594298,229.05,4,2,2,SQL / BI,Analysis / Research,SQL / BI Tools,,,True,High,952.0,
8,c0c84690-a4a2-4a90-b172-087644c99041,Expert Data Engineer Tips.,2023-02-14,2023-02-14 14:42:10.408931,2023-02-14 14:42:25.038141,0.24,2,1,1,General / Other,General Inquiry,General Purpose,,,True,Low,43.0,
9,c8352ed4-2e77-4bd2-b4ed-ed0f10d8d865,Identify non-buying customers.,2023-02-14,2023-02-14 16:33:29.015675,2023-02-14 16:33:58.605748,0.49,2,1,1,SQL / BI,How-to / Question,SQL / BI Tools,,,True,High,2031.0,



Top 10 primary_category:
primary_category
General / Other          319
SQL / BI                 234
Data Engineering         151
Academic / Study         150
AWS / Cloud              114
Python / Programming     104
Translation / Writing     91
Personal / Advice         51
Machine Learning / AI     41
Name: count, dtype: int64

Top 10 primary_type:
primary_type
How-to / Question            763
Troubleshooting              152
General Inquiry              112
Analysis / Research           93
Build / Design                89
Writing / Translation         37
Interview Prep / Learning      9
Name: count, dtype: int64

Top 10 primary_tool:
primary_tool
General Purpose             314
Apache Spark                116
Study / Coursework          114
Power BI                     90
Writing / Translation        80
SQL Server / SSIS / SSRS     79
SQL / BI Tools               59
Personal / Daily Life        47
Python                       47
Apache Airflow               35
Name: count, dtype: int